# LeetCode 463 — Island Perimeter bằng Python

**Link đề:** https://leetcode.com/problems/island-perimeter/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix (ma trận / lưới 2 chiều), DFS, BFS

```python
class Solution:
    def islandPerimeter(self, grid: list[list[int]]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu cách biểu diễn bản đồ bằng **list 2 chiều** `grid[r][c]` và duyệt bằng hai vòng `for` lồng nhau.
2. Biết kiểm tra **ô hàng xóm** (trên/dưới/trái/phải) an toàn khi ở biên lưới, dùng list hướng `DIRS`.
3. Đi từ cách "đếm cạnh lộ ra của từng ô" → công thức `4·đất − 2·cạnh chung` → DFS/BFS loang từ một ô đất.
4. Trace từng ô, chứng minh vì sao công thức đúng, phân tích Time/Space.
5. Nhận dạng pattern "lưới + hàng xóm 4 hướng", làm 5 bài tập có gợi ý và đáp án ở cuối.

**Cách học:** đọc markdown → đoán output cell kế tiếp → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho lưới `grid` kích thước `row × col`: `grid[i][j] = 1` là **đất**, `0` là **nước**. Các ô nối nhau theo chiều ngang/dọc (không chéo). Lưới được bao quanh bởi nước và có **đúng một hòn đảo** (một hoặc nhiều ô đất liền nhau). Đảo **không có hồ** (nước bên trong không thông ra nước bên ngoài). Mỗi ô là hình vuông cạnh 1. Tính **chu vi** của đảo.

| `grid` | Output |
|---|:---:|
| `[[0,1,0,0],[1,1,1,0],[0,1,0,0],[1,1,0,0]]` | 16 |
| `[[1]]` | 4 |
| `[[1,0]]` | 4 |

**Constraints:** `1 <= row, col <= 100`, `grid[i][j]` là 0 hoặc 1, đúng một đảo.

Chu vi = tổng số **cạnh đơn vị** của ô đất mà phía bên kia là **nước hoặc mép lưới**.

In [1]:
grid = [[0, 1, 0, 0],
        [1, 1, 1, 0],
        [0, 1, 0, 0],
        [1, 1, 0, 0]]

def show(grid):
    for row in grid:
        print(" ".join("#" if v == 1 else "." for v in row))

show(grid)

. # . .
# # # .
. # . .
# # . .


Câu hỏi đặt ra trước khi code:
- Một ô đất đứng một mình có chu vi bao nhiêu? Hai ô đất cạnh nhau thì sao?
- Khi một ô nằm ở mép lưới, làm sao "nhìn sang bên cạnh" mà không bị lỗi chỉ số?
- Có cần biết hình dạng đảo (loang) không, hay chỉ cần đếm?

## 2. Nền tảng Python — list 2 chiều

### 2.1. Truy cập `grid[r][c]`

`grid` là **list của các list**. `grid[r]` là hàng thứ `r` (một list), `grid[r][c]` là ô ở hàng `r`, cột `c`. Số hàng là `len(grid)`, số cột là `len(grid[0])`.

In [2]:
print("số hàng:", len(grid), "| số cột:", len(grid[0]))
print("hàng 1:", grid[1])
print("ô (1, 2):", grid[1][2])
print("ô (0, 0):", grid[0][0])

số hàng: 4 | số cột: 4
hàng 1: [1, 1, 1, 0]
ô (1, 2): 1
ô (0, 0): 0


### 2.2. Duyệt mọi ô bằng hai vòng lặp lồng nhau

Vòng ngoài chạy qua hàng `r`, vòng trong chạy qua cột `c`. Tổng số lần lặp = `rows × cols`.

In [3]:
small = [[1, 0],
         [1, 1]]
for r in range(len(small)):
    for c in range(len(small[0])):
        print(f"({r},{c}) = {small[r][c]}", end="   ")
    print()   # xuống dòng sau mỗi hàng

(0,0) = 1   (0,1) = 0   
(1,0) = 1   (1,1) = 1   


### 2.3. Bốn hàng xóm và list hướng `DIRS`

Ô `(r, c)` có 4 hàng xóm: trên `(r-1, c)`, dưới `(r+1, c)`, trái `(r, c-1)`, phải `(r, c+1)`. Ta viết gọn thành list các **độ lệch** (dr, dc) rồi lặp:

```python
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
for dr, dc in DIRS:
    nr, nc = r + dr, c + dc
```

`for dr, dc in DIRS` là **tách bộ** (tuple unpacking): mỗi phần tử `(-1, 0)` được tách thành `dr = -1`, `dc = 0`.

In [4]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
names = ["trên", "dưới", "trái", "phải"]
r, c = 1, 1
for (dr, dc), name in zip(DIRS, names):
    print(f"{name:>5}: ({r + dr}, {c + dc})")

 trên: (0, 1)
 dưới: (2, 1)
 trái: (1, 0)
 phải: (1, 2)


### 2.4. Kiểm tra biên — tránh "chỉ số âm" nguy hiểm

Ô `(0, 0)` có hàng xóm trên là `(-1, 0)`. Python **không báo lỗi** với chỉ số `-1` mà lấy **hàng cuối cùng**! Đây là bug âm thầm. Phải kiểm tra `0 <= nr < rows and 0 <= nc < cols` trước khi truy cập.

Python cho phép viết so sánh chuỗi `0 <= nr < rows`, nghĩa là `0 <= nr and nr < rows`.

In [5]:
rows, cols = len(grid), len(grid[0])
print("grid[-1][0] =", grid[-1][0], "<- thực ra là ô (3, 0), KHÔNG phải ô ngoài lưới!")

def inside(r, c):
    return 0 <= r < rows and 0 <= c < cols

print(inside(-1, 0), inside(0, 0), inside(3, 3), inside(4, 0))

grid[-1][0] = 1 <- thực ra là ô (3, 0), KHÔNG phải ô ngoài lưới!
False True True False


## 3. Nhập môn bằng tay — mỗi ô đất góp bao nhiêu cạnh?

- Một ô đất đứng riêng: **4** cạnh.
- Hai ô đất cạnh nhau: mỗi ô mất 1 cạnh (cạnh chung nằm "bên trong" đảo) → `4 + 4 − 2 = 6`.
- Ba ô thẳng hàng: `3·4 − 2·2 = 8`.

Quy tắc: với mỗi ô đất, **mỗi phía** là nước hoặc mép lưới → cộng 1 vào chu vi.

In [6]:
for g in ([[1]], [[1, 1]], [[1, 1, 1]], [[1, 1], [1, 1]]):
    land = sum(sum(row) for row in g)
    print(g, "-> số ô đất =", land)

[[1]] -> số ô đất = 1
[[1, 1]] -> số ô đất = 2
[[1, 1, 1]] -> số ô đất = 3
[[1, 1], [1, 1]] -> số ô đất = 4


`sum(sum(row) for row in g)`: tổng của từng hàng (đếm số 1 trong hàng), rồi cộng các hàng lại → tổng số ô đất. Với lưới `[[1,1],[1,1]]` có 4 ô đất và 4 cạnh chung → chu vi `16 − 8 = 8`.

## 4. Cách 1 — Đếm cạnh lộ ra của từng ô đất (trực giác)

Với mỗi ô đất, nhìn 4 hướng: nếu hàng xóm **ra ngoài lưới** hoặc **là nước** thì cạnh đó thuộc chu vi.

In [7]:
def perimeter_count_sides(grid: list[list[int]]) -> int:
    rows, cols = len(grid), len(grid[0])
    perimeter = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if not (0 <= nr < rows and 0 <= nc < cols) or grid[nr][nc] == 0:
                        perimeter += 1
    return perimeter

print(perimeter_count_sides(grid))      # 16
print(perimeter_count_sides([[1]]))     # 4
print(perimeter_count_sides([[1, 0]]))  # 4

16
4
4


Chú ý thứ tự trong điều kiện: `not inside(...) or grid[nr][nc] == 0`. Nhờ **ngắn mạch** của `or`, nếu ô ra ngoài lưới thì vế phải **không được tính** → không truy cập chỉ số sai.

## 5. TRACE từng ô đất

Với lưới ví dụ (`#` là đất):

```text
. # . .
# # # .
. # . .
# # . .
```

| ô đất | trên | dưới | trái | phải | cạnh lộ ra |
|---|---|---|---|---|---|
| (0,1) | mép | đất | nước | nước | 3 |
| (1,0) | nước | nước | mép | đất | 3 |
| (1,1) | đất | đất | đất | đất | 0 |
| (1,2) | nước | nước | đất | nước | 3 |
| (2,1) | đất | đất | nước | nước | 2 |
| (3,0) | nước | mép | mép | đất | 3 |
| (3,1) | đất | mép | đất | nước | 2 |
| **tổng** | | | | | **16** |

In [8]:
def trace_perimeter(grid):
    rows, cols = len(grid), len(grid[0])
    total = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != 1:
                continue
            desc = []
            exposed = 0
            for (dr, dc), name in zip(DIRS, ["trên", "dưới", "trái", "phải"]):
                nr, nc = r + dr, c + dc
                if not (0 <= nr < rows and 0 <= nc < cols):
                    desc.append(f"{name}=mép"); exposed += 1
                elif grid[nr][nc] == 0:
                    desc.append(f"{name}=nước"); exposed += 1
                else:
                    desc.append(f"{name}=đất")
            total += exposed
            print(f"ô ({r},{c}): {', '.join(desc):<40} -> +{exposed}, tổng = {total}")
    return total

trace_perimeter(grid)

ô (0,1): trên=mép, dưới=đất, trái=nước, phải=nước -> +3, tổng = 3
ô (1,0): trên=nước, dưới=nước, trái=mép, phải=đất -> +3, tổng = 6
ô (1,1): trên=đất, dưới=đất, trái=đất, phải=đất   -> +0, tổng = 6
ô (1,2): trên=nước, dưới=nước, trái=đất, phải=nước -> +3, tổng = 9
ô (2,1): trên=đất, dưới=đất, trái=nước, phải=nước -> +2, tổng = 11
ô (3,0): trên=nước, dưới=mép, trái=mép, phải=đất  -> +3, tổng = 14
ô (3,1): trên=đất, dưới=mép, trái=đất, phải=nước  -> +2, tổng = 16


16

`continue` bỏ qua phần còn lại của vòng lặp hiện tại và sang ô kế tiếp — giúp code ít thụt lề hơn. `', '.join(desc)` nối các chuỗi trong list bằng dấu phẩy.

## 6. Bước đột phá — công thức `4·đất − 2·cạnh chung`

Mỗi ô đất góp 4 cạnh. Mỗi **cặp ô đất kề nhau** có một cạnh chung, cạnh này bị đếm ở **cả hai** ô nhưng không thuộc chu vi → trừ 2.

> `chu vi = 4 × (số ô đất) − 2 × (số cặp đất kề nhau)`

Để đếm mỗi cặp **đúng một lần**, mỗi ô đất chỉ cần nhìn **xuống dưới** và **sang phải** (hoặc lên trên và sang trái). Không cần kiểm tra biên phức tạp nữa, chỉ cần `r + 1 < rows` và `c + 1 < cols`.

Kiểm tra với lưới ví dụ: 7 ô đất, 6 cặp kề → `28 − 12 = 16` ✔.

## 7. Lời giải chính — mổ xẻ từng đoạn

### 7.1. Đếm ô đất
```python
if grid[r][c] == 1:
    land += 1
```
### 7.2. Đếm cặp kề — chỉ nhìn dưới và phải
```python
    if r + 1 < rows and grid[r + 1][c] == 1: shared += 1
    if c + 1 < cols and grid[r][c + 1] == 1: shared += 1
```
### 7.3. Ghép công thức
```python
return 4 * land - 2 * shared
```

In [9]:
def perimeter_formula(grid: list[list[int]]) -> int:
    rows, cols = len(grid), len(grid[0])
    land = shared = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                land += 1
                if r + 1 < rows and grid[r + 1][c] == 1:
                    shared += 1
                if c + 1 < cols and grid[r][c + 1] == 1:
                    shared += 1
    return 4 * land - 2 * shared

print(perimeter_formula(grid), perimeter_formula([[1]]), perimeter_formula([[1, 0]]))

16 4 4


In [10]:
def trace_formula(grid):
    rows, cols = len(grid), len(grid[0])
    land = shared = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] != 1:
                continue
            land += 1
            pairs = []
            if r + 1 < rows and grid[r + 1][c] == 1:
                shared += 1; pairs.append(f"dưới ({r+1},{c})")
            if c + 1 < cols and grid[r][c + 1] == 1:
                shared += 1; pairs.append(f"phải ({r},{c+1})")
            print(f"ô ({r},{c}): land={land}, cặp mới: {pairs or 'không'}, shared={shared}")
    print(f"chu vi = 4*{land} - 2*{shared} = {4 * land - 2 * shared}")

trace_formula(grid)

ô (0,1): land=1, cặp mới: ['dưới (1,1)'], shared=1
ô (1,0): land=2, cặp mới: ['phải (1,1)'], shared=2
ô (1,1): land=3, cặp mới: ['dưới (2,1)', 'phải (1,2)'], shared=4
ô (1,2): land=4, cặp mới: không, shared=4
ô (2,1): land=5, cặp mới: ['dưới (3,1)'], shared=5
ô (3,0): land=6, cặp mới: ['phải (3,1)'], shared=6
ô (3,1): land=7, cặp mới: không, shared=6
chu vi = 4*7 - 2*6 = 16


`pairs or 'không'`: nếu list `pairs` rỗng (được coi là `False`) thì biểu thức trả `'không'`, ngược lại trả chính `pairs`.

## 8. Cách 3 — DFS loang từ một ô đất

Một hướng "đồ thị": tìm một ô đất, rồi **loang** (Depth-First Search — tìm kiếm theo chiều sâu) sang các ô đất kề. Khi từ một ô đất bước sang nước hoặc ra ngoài lưới → cộng 1 vào chu vi. Dùng `set` `visited` để không thăm một ô hai lần.

Cách này dư thừa cho bài này (vì đếm toàn lưới đã đủ), nhưng là **nền tảng** cho các bài lưới khó hơn (Number of Islands, Max Area of Island) — đặc biệt khi có **nhiều** đảo và chỉ cần chu vi của một đảo.

Ta dùng **stack** (list với `append`/`pop`) thay cho đệ quy để tránh giới hạn độ sâu đệ quy của Python (mặc định khoảng 1000, lưới 100×100 có thể tới 10.000 ô).

In [11]:
stack = []
stack.append((0, 1)); stack.append((1, 1))
print("pop:", stack.pop(), "| còn lại:", stack)   # lấy phần tử vào sau cùng (LIFO)

def perimeter_dfs(grid: list[list[int]]) -> int:
    rows, cols = len(grid), len(grid[0])
    start = next((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == 1)
    visited = {start}
    stack = [start]
    perimeter = 0
    while stack:
        r, c = stack.pop()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < rows and 0 <= nc < cols) or grid[nr][nc] == 0:
                perimeter += 1                  # bước ra nước / mép -> một cạnh chu vi
            elif (nr, nc) not in visited:
                visited.add((nr, nc))
                stack.append((nr, nc))
    return perimeter

print(perimeter_dfs(grid), perimeter_dfs([[1]]), perimeter_dfs([[1, 0]]))

pop: (1, 1) | còn lại: [(0, 1)]
16 4 4


Giải thích: `next(generator)` lấy phần tử đầu tiên — ở đây là ô đất đầu tiên theo thứ tự hàng/cột. `visited = {start}` tạo một `set` có sẵn một phần tử (tuple tọa độ dùng được làm phần tử của set vì tuple **không đổi** — hashable).

### 8.1. Cách 4 — BFS với `deque`

Thay stack bằng **hàng đợi** `collections.deque` (`append` + `popleft`) ta được BFS (Breadth-First Search — loang theo lớp). Kết quả chu vi giống hệt; chỉ thứ tự thăm khác.

In [12]:
from collections import deque

def perimeter_bfs(grid: list[list[int]]) -> int:
    rows, cols = len(grid), len(grid[0])
    start = next((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == 1)
    visited = {start}
    queue = deque([start])
    perimeter = 0
    while queue:
        r, c = queue.popleft()                 # lấy phần tử vào trước (FIFO)
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < rows and 0 <= nc < cols) or grid[nr][nc] == 0:
                perimeter += 1
            elif (nr, nc) not in visited:
                visited.add((nr, nc))
                queue.append((nr, nc))
    return perimeter

print(perimeter_bfs(grid))

16


## 9. Code nộp LeetCode (Python 3)

Bản chính: công thức `4·đất − 2·cạnh chung` (ngắn, không cần bộ nhớ phụ).

In [13]:
class Solution:
    def islandPerimeter(self, grid: list[list[int]]) -> int:
        rows, cols = len(grid), len(grid[0])
        land = shared = 0
        for r in range(rows):
            for c in range(cols):
                if grid[r][c] == 1:
                    land += 1
                    if r + 1 < rows and grid[r + 1][c] == 1:
                        shared += 1
                    if c + 1 < cols and grid[r][c + 1] == 1:
                        shared += 1
        return 4 * land - 2 * shared

sol = Solution()
print(sol.islandPerimeter([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]]))  # 16
print(sol.islandPerimeter([[1]]))      # 4
print(sol.islandPerimeter([[1, 0]]))   # 4

16
4
4


## 10. Vì sao đúng?

**Cách đếm cạnh lộ ra:** chu vi theo định nghĩa là tổng các cạnh đơn vị ngăn cách **đất** với **không-đất** (nước hoặc ngoài lưới). Mỗi cạnh như vậy thuộc đúng **một** ô đất (phía bên kia không phải đất), nên duyệt mọi ô đất và mọi hướng đếm mỗi cạnh chu vi **đúng một lần**. Điều kiện "không có hồ" bảo đảm không có nước bị bao bên trong — nếu có hồ, cạnh hồ cũng sẽ bị cộng vào (vẫn đúng nếu đề định nghĩa chu vi gồm cả bờ hồ; ở đây đề loại trừ trường hợp này).

**Công thức:** mỗi ô đất có 4 cạnh → tổng `4·land` lần "cạnh-thuộc-ô". Cạnh nằm giữa hai ô đất bị đếm 2 lần và không thuộc chu vi; các cạnh còn lại thuộc chu vi đúng 1 lần. Vậy `chu vi = 4·land − 2·shared`. Mỗi cặp kề nằm ngang được đếm duy nhất bởi ô bên **trái** (nhìn phải), cặp nằm dọc bởi ô **trên** (nhìn xuống) → `shared` đếm mỗi cặp đúng một lần.

**DFS/BFS:** loang thăm đúng mọi ô của đảo (đảo liên thông), mỗi ô một lần; với mỗi ô xét đủ 4 hướng → giống cách 1 nhưng chỉ trên các ô của đảo.

## 11. Phân tích Time / Space

Gọi `R = rows`, `C = cols`.

| Cách | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|---|---|
| 1. Đếm cạnh lộ ra | `O(R·C)` (4 phép kiểm tra / ô) | `O(1)` | trực giác |
| 2. `4·đất − 2·chung` | `O(R·C)` (2 phép kiểm tra / ô) | `O(1)` | **gọn nhất** |
| 3. DFS (stack) | `O(R·C)` (tìm ô đầu + loang) | `O(số ô đất)` | nền tảng bài lưới |
| 4. BFS (deque) | `O(R·C)` | `O(số ô đất)` | như DFS |

Với `R, C <= 100` → tối đa 10.000 ô, mọi cách đều rất nhanh.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual}")

cases = [
    ("đề 1", [[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]], 16),
    ("đề 2", [[1]], 4),
    ("đề 3", [[1, 0]], 4),
    ("một hàng đất", [[1, 1, 1, 1]], 10),
    ("một cột đất", [[1], [1], [1]], 8),
    ("khối vuông 2x2", [[1, 1], [1, 1]], 8),
    ("toàn đất 3x3", [[1] * 3 for _ in range(3)], 12),
    ("chữ L", [[1, 0], [1, 0], [1, 1]], 10),
    ("đất ở góc", [[0, 0], [0, 1]], 4),
]
sol = Solution()
for name, g, expected in cases:
    check(name, sol.islandPerimeter(g), expected)

PASS  đề 1: 16
PASS  đề 2: 4
PASS  đề 3: 4
PASS  một hàng đất: 10
PASS  một cột đất: 8
PASS  khối vuông 2x2: 8
PASS  toàn đất 3x3: 12
PASS  chữ L: 10
PASS  đất ở góc: 4


Để kiểm đối chiếu ngẫu nhiên ta cần sinh **đúng một đảo liên thông**. Cách đơn giản: bắt đầu từ một ô, lặp nhiều lần "chọn một ô đất có sẵn rồi biến một hàng xóm hợp lệ thành đất" — đảo luôn liên thông. (Đảo sinh ra có thể có hồ; các cách 1, 2, 3, 4 vẫn phải cho cùng kết quả vì đều đếm cạnh đất–không đất.)

In [15]:
import random
random.seed(463)

def random_island(rows, cols, steps):
    g = [[0] * cols for _ in range(rows)]
    r, c = random.randrange(rows), random.randrange(cols)
    g[r][c] = 1
    cells = [(r, c)]
    for _ in range(steps):
        r, c = random.choice(cells)
        dr, dc = random.choice(DIRS)
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and g[nr][nc] == 0:
            g[nr][nc] = 1
            cells.append((nr, nc))
    return g

methods = [perimeter_count_sides, perimeter_formula, perimeter_dfs, perimeter_bfs]
for _ in range(500):
    g = random_island(random.randint(1, 8), random.randint(1, 8), random.randint(0, 30))
    results = {f(g) for f in methods}
    assert len(results) == 1, (g, results)
print("500 đảo ngẫu nhiên: 4 cách cho cùng chu vi")
example = random_island(5, 6, 12)
show(example)
print("chu vi:", Solution().islandPerimeter(example))

500 đảo ngẫu nhiên: 4 cách cho cùng chu vi
. . . . . .
. . . . . .
# # . . . .
# # . . . .
# # # . . .
chu vi: 12


Lưu ý `[[0] * cols for _ in range(rows)]` — **không** dùng `[[0] * cols] * rows` (xem lỗi 3 bên dưới).

## 13. Lỗi người mới hay mắc

### Lỗi 1: không kiểm tra biên → chỉ số âm lấy nhầm ô phía bên kia lưới

In [16]:
def wrong_no_bounds(grid):
    rows, cols = len(grid), len(grid[0])
    p = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    try:
                        if grid[nr][nc] == 0:   # SAI: nr=-1 lấy hàng cuối, không báo lỗi
                            p += 1
                    except IndexError:          # chỉ bắt được khi vượt quá phía dưới/phải
                        p += 1
    return p

g = [[1, 0], [1, 0], [1, 1]]
print("sai :", wrong_no_bounds(g))
print("đúng:", perimeter_formula(g))

sai : 8
đúng: 10


### Lỗi 2: đếm cặp kề ở cả 4 hướng nhưng vẫn trừ `2 ×`
Nếu mỗi ô nhìn cả 4 hướng để đếm hàng xóm đất, mỗi cặp bị đếm **hai lần**; khi đó công thức phải là `4·land − neighbors` (trừ 1 lần mỗi lượt đếm), không phải `− 2·neighbors`.

In [17]:
def wrong_double_count(grid):
    rows, cols = len(grid), len(grid[0])
    land = nb = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                land += 1
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                        nb += 1
    return 4 * land - 2 * nb       # SAI: nb đã đếm mỗi cặp 2 lần

print("sai :", wrong_double_count(grid))
print("đúng:", perimeter_formula(grid))

sai : 4
đúng: 16


### Lỗi 3: tạo lưới bằng `[[0] * cols] * rows`
Phép nhân list ngoài **sao chép tham chiếu**: mọi hàng là **cùng một** list. Sửa một ô là sửa cả cột!

In [18]:
bad = [[0] * 3] * 2
bad[0][0] = 1
print("tạo sai :", bad)
good = [[0] * 3 for _ in range(2)]
good[0][0] = 1
print("tạo đúng:", good)

tạo sai : [[1, 0, 0], [1, 0, 0]]
tạo đúng: [[1, 0, 0], [0, 0, 0]]


### Lỗi 4: nhầm `len(grid)` và `len(grid[0])`
Số hàng là `len(grid)`, số cột là `len(grid[0])`. Với lưới không vuông (như `[[1, 0]]`: 1 hàng, 2 cột), nhầm sẽ gây `IndexError` hoặc bỏ sót ô.

### Lỗi 5: DFS đệ quy trên lưới lớn
Đảo 100×100 toàn đất → độ sâu đệ quy có thể tới 10.000 > giới hạn mặc định (~1000) → `RecursionError`. Dùng stack tường minh như mục 8, hoặc `sys.setrecursionlimit`.

## 14. Mẹo nhận dạng pattern

- Đề cho **lưới 0/1** và hỏi về **cạnh / hàng xóm 4 hướng** → `DIRS` + kiểm tra biên.
- Chỉ cần **đếm** (chu vi, diện tích tổng) → duyệt toàn lưới là đủ, không cần loang.
- Cần phân biệt **từng vùng liên thông** (nhiều đảo, đảo lớn nhất, …) → DFS/BFS + `visited`.
- Mẹo "đếm mỗi cặp một lần": chỉ nhìn **phải** và **dưới**.

Bài cùng họ: 200 Number of Islands, 695 Max Area of Island, 733 Flood Fill, 892 Surface Area of 3D Shapes, 1020 Number of Enclaves, 827 Making A Large Island.

## 15. Biến thể

### 15.1. Nhiều đảo — chu vi **lớn nhất** trong các đảo

In [19]:
def max_island_perimeter(grid):
    rows, cols = len(grid), len(grid[0])
    seen = set()
    best = 0
    for sr in range(rows):
        for sc in range(cols):
            if grid[sr][sc] == 1 and (sr, sc) not in seen:
                seen.add((sr, sc))
                stack = [(sr, sc)]
                per = 0
                while stack:
                    r, c = stack.pop()
                    for dr, dc in DIRS:
                        nr, nc = r + dr, c + dc
                        if not (0 <= nr < rows and 0 <= nc < cols) or grid[nr][nc] == 0:
                            per += 1
                        elif (nr, nc) not in seen:
                            seen.add((nr, nc))
                            stack.append((nr, nc))
                best = max(best, per)
    return best

print(max_island_perimeter([[1, 0, 1, 1],
                            [0, 0, 1, 1],
                            [1, 0, 0, 0]]))   # đảo 2x2 có chu vi 8

8


### 15.2. Surface Area 3D (ý tưởng LeetCode 892)
Mỗi ô có `v` khối lập phương chồng lên nhau. Diện tích bề mặt = `Σ (4v + 2 nếu v > 0) − 2·Σ min(cặp kề)` — cùng mẫu "tổng mỗi phần tử − 2 × phần chung" như bài này.

In [20]:
def surface_area(grid):
    n, m = len(grid), len(grid[0])
    total = 0
    for r in range(n):
        for c in range(m):
            v = grid[r][c]
            if v:
                total += 4 * v + 2
            if r + 1 < n:
                total -= 2 * min(v, grid[r + 1][c])
            if c + 1 < m:
                total -= 2 * min(v, grid[r][c + 1])
    return total

print(surface_area([[1, 2], [3, 4]]))   # 34
print(surface_area([[2]]))              # 10

34
10


## 16. Bài tập tự luyện (tăng dần)

> Viết code vào các hàm TODO; test để ở dạng comment. Gợi ý ở cuối mục này, đáp án ở mục 17.

### Bài 1 — Tính tay
Lưới `[[1,1,0],[0,1,1]]`. Đếm số ô đất, số cặp kề, rồi tính chu vi bằng công thức.

### Bài 2 — Hoàn thiện cách đếm cạnh lộ ra

In [21]:
def ex2_perimeter(grid: list[list[int]]) -> int:
    # TODO 1: rows, cols
    # TODO 2: duyệt mọi ô đất
    # TODO 3: với mỗi hướng trong DIRS, nếu ra ngoài hoặc là nước -> +1
    pass

# assert ex2_perimeter([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]]) == 16
# assert ex2_perimeter([[1, 0]]) == 4

### Bài 3 — Diện tích đảo (số ô đất) bằng một dòng

In [22]:
def ex3_area(grid: list[list[int]]) -> int:
    # TODO: dùng sum lồng nhau
    pass

# assert ex3_area([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]]) == 7

### Bài 4 — Number of Islands (LeetCode 200, bản số nguyên)
Đếm số đảo trong lưới 0/1 (có thể nhiều đảo).

In [23]:
def ex4_num_islands(grid: list[list[int]]) -> int:
    # TODO: duyệt mọi ô; gặp đất chưa thăm -> tăng đếm và loang hết đảo đó
    pass

# assert ex4_num_islands([[1, 1, 0, 0], [1, 0, 0, 1], [0, 0, 1, 1]]) == 2
# assert ex4_num_islands([[0]]) == 0

### Bài 5 — Mini challenge: chu vi khi đảo **có hồ**
Định nghĩa "chu vi ngoài" = chỉ tính cạnh tiếp xúc với nước **thông ra ngoài lưới** (không tính bờ hồ). Ví dụ lưới 3×3 toàn đất trừ ô giữa là nước: chu vi ngoài = 12 (còn chu vi đầy đủ là 16).

In [24]:
def ex5_outer_perimeter(grid: list[list[int]]) -> int:
    # TODO: loang nước từ ngoài vào (thêm viền nước bao quanh lưới), đếm cạnh chạm đất
    pass

# assert ex5_outer_perimeter([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == 12
# assert ex5_outer_perimeter([[1]]) == 4

### Gợi ý 3 mức (chỉ mở sau khi tự thử)

**Bài 1** — Mức 1: đếm ô có giá trị 1. Mức 2: chỉ nhìn phải và dưới. Mức 3: 4 ô đất, 3 cặp → `16 − 6`.

**Bài 2** — Mức 1: copy cấu trúc hai vòng `for`. Mức 2: `nr, nc = r + dr, c + dc`. Mức 3: `if not (0 <= nr < rows and 0 <= nc < cols) or grid[nr][nc] == 0: p += 1`.

**Bài 3** — Mức 1: `sum(row)` đếm đất trong một hàng. Mức 2: cộng các hàng. Mức 3: `sum(sum(row) for row in grid)`.

**Bài 4** — Mức 1: cần `visited`. Mức 2: dùng stack như `perimeter_dfs`. Mức 3: mỗi lần bắt đầu loang mới thì `count += 1`.

**Bài 5** — Mức 1: thêm viền nước bao quanh để mọi nước ngoài liên thông. Mức 2: loang **nước** từ góc `(0, 0)` của lưới mới. Mức 3: mỗi lần từ ô nước nhìn sang ô đất → cộng 1.

## 17. Đáp án tham khảo

**Bài 1:** đất = 4 (`(0,0),(0,1),(1,1),(1,2)`), cặp kề = 3 → chu vi `16 − 6 = 10`.

In [25]:
def ans2_perimeter(grid):
    rows, cols = len(grid), len(grid[0])
    p = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 1:
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if not (0 <= nr < rows and 0 <= nc < cols) or grid[nr][nc] == 0:
                        p += 1
    return p

def ans3_area(grid):
    return sum(sum(row) for row in grid)

def ans4_num_islands(grid):
    rows, cols = len(grid), len(grid[0])
    seen = set()
    count = 0
    for sr in range(rows):
        for sc in range(cols):
            if grid[sr][sc] == 1 and (sr, sc) not in seen:
                count += 1
                seen.add((sr, sc))
                stack = [(sr, sc)]
                while stack:
                    r, c = stack.pop()
                    for dr, dc in DIRS:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1 and (nr, nc) not in seen:
                            seen.add((nr, nc))
                            stack.append((nr, nc))
    return count

def ans5_outer_perimeter(grid):
    rows, cols = len(grid) + 2, len(grid[0]) + 2
    padded = [[0] * cols] + [[0] + row + [0] for row in grid] + [[0] * cols]
    seen = {(0, 0)}
    stack = [(0, 0)]
    p = 0
    while stack:
        r, c = stack.pop()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if not (0 <= nr < rows and 0 <= nc < cols):
                continue
            if padded[nr][nc] == 1:
                p += 1                       # cạnh giữa nước ngoài và đất
            elif (nr, nc) not in seen:
                seen.add((nr, nc))
                stack.append((nr, nc))
    return p

assert perimeter_formula([[1, 1, 0], [0, 1, 1]]) == 10                 # bài 1
assert ans2_perimeter([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]]) == 16
assert ans2_perimeter([[1, 0]]) == 4
assert ans3_area([[0, 1, 0, 0], [1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 0, 0]]) == 7
assert ans4_num_islands([[1, 1, 0, 0], [1, 0, 0, 1], [0, 0, 1, 1]]) == 2
assert ans4_num_islands([[0]]) == 0
assert ans5_outer_perimeter([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == 12
assert ans5_outer_perimeter([[1]]) == 4
assert perimeter_formula([[1, 1, 1], [1, 0, 1], [1, 1, 1]]) == 16   # chu vi đầy đủ gồm cả bờ hồ
print("Đáp án bài 1–5: PASS")

Đáp án bài 1–5: PASS


Trong đáp án bài 5, `[[0] * cols]` ở đầu và cuối chỉ dùng để **đọc** (không sửa) nên việc nhân list không gây lỗi chia sẻ tham chiếu; `[0] + row + [0]` nối list để thêm cột nước hai bên.

## 18. Template nhớ nhanh + checklist submit

```python
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
rows, cols = len(grid), len(grid[0])
land = shared = 0
for r in range(rows):
    for c in range(cols):
        if grid[r][c] == 1:
            land += 1
            if r + 1 < rows and grid[r + 1][c] == 1: shared += 1   # chỉ nhìn dưới
            if c + 1 < cols and grid[r][c + 1] == 1: shared += 1   # và phải
return 4 * land - 2 * shared
```

**Checklist trước khi submit**
- [ ] Có kiểm tra biên trước khi đọc `grid[nr][nc]` (đặc biệt chỉ số âm)?
- [ ] `rows = len(grid)`, `cols = len(grid[0])` — không nhầm?
- [ ] Nếu dùng công thức: mỗi cặp kề chỉ đếm **một** lần?
- [ ] Ca một ô `[[1]]` → 4, lưới một hàng/một cột?
- [ ] Nếu loang: dùng `visited`, tránh đệ quy quá sâu?